# WM 2026 Match Predictor – Datenanalyse & Modell-Evaluation

Zentrales Notebook: Datenpipeline-Status, explorative Datenanalyse (EDA), Feature-Auswahl
und Auswertung/Test des trainierten Random-Forest-Modells.

Voraussetzung: `fetch_data.py`, `fetch_match_stats.py`, `build_features.py` und
`train_model.py` wurden bereits ausgeführt.

---

### **1) Setup**
Imports, Pfade und ein kleiner Encoding-Helper (ältere Fixture-Dateien wurden vor einem
Fix teils mit `cp1252` statt UTF-8 gespeichert).

In [ ]:
import sys
import json
import csv
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.metrics import classification_report, confusion_matrix

sys.path.insert(0, str(Path.cwd().parent / "src"))
from config import DATA_RAW_DIR, DATA_PROCESSED_DIR, PROJECT_ROOT, WM2026_TEAMS

STATS_DIR = DATA_RAW_DIR / "stats"
FEATURES_FILE = DATA_PROCESSED_DIR / "match_features.csv"
MODELS_DIR = PROJECT_ROOT / "models"

pd.set_option("display.max_columns", None)


def _read_json_robust(path: Path):
    """Liest JSON ein, mit Fallback auf cp1252 (ältere Dateien vor Encoding-Fix)."""
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except UnicodeDecodeError:
        return json.loads(path.read_text(encoding="cp1252"))

---

### **2) Datenpipeline-Status**
Fixtures- und Match-Stats-Fortschritt pro Team, plus Abdeckung der einzelnen Statistik-Typen
(Ballbesitz, Ecken etc.) über alle Spiele hinweg.

In [ ]:
def analyze_data_status():
    stats_files = list(STATS_DIR.glob("*.json"))

    existing_stats_ids = {
        int(stats_file.stem)
        for stats_file in stats_files
        if stats_file.stem.isdigit()
    }

    # ==================================================
    # 1. FIXTURE- UND DOWNLOAD-STATUS
    # ==================================================
    team_rows = []
    total_fixtures = 0
    total_unique_ids = set()

    for fifa_name in WM2026_TEAMS:
        fixtures_file = (
            DATA_RAW_DIR
            / f"fixtures_{fifa_name.replace(' ', '_')}.json"
        )

        if not fixtures_file.exists():
            team_rows.append((fifa_name, "FEHLT", "-", "-", "\u274c"))
            continue

        if fixtures_file.stat().st_size == 0:
            team_rows.append((fifa_name, "LEER", "-", "-", "\u26a0\ufe0f"))
            continue

        try:
            fixtures = _read_json_robust(fixtures_file)
        except (json.JSONDecodeError, UnicodeDecodeError):
            team_rows.append((fifa_name, "KAPUTT", "-", "-", "\u274c"))
            continue

        fixture_ids = [
            fixture.get("fixture", {}).get("id")
            for fixture in fixtures
        ]
        fixture_ids = [fid for fid in fixture_ids if fid is not None]

        n_games = len(fixture_ids)
        n_stats_done = sum(fid in existing_stats_ids for fid in fixture_ids)
        progress = 100 * n_stats_done / n_games if n_games > 0 else 0

        if progress == 100:
            status = "\u2705"
        elif progress > 0:
            status = "\U0001f7e1"
        else:
            status = "\u26aa"

        team_rows.append((fifa_name, str(n_games), str(n_stats_done), f"{progress:.0f}%", status))
        total_fixtures += n_games
        total_unique_ids.update(fixture_ids)

    available_stats_ids = existing_stats_ids & total_unique_ids
    overall_progress = 100 * len(available_stats_ids) / len(total_unique_ids) if total_unique_ids else 0
    valid_teams = sum(row[1] not in ("FEHLT", "LEER", "KAPUTT") for row in team_rows)
    problem_teams = [row[0] for row in team_rows if row[1] in ("FEHLT", "LEER", "KAPUTT")]

    # ==================================================
    # 2. STATISTIKABDECKUNG
    # ==================================================
    type_full_coverage = defaultdict(int)
    valid_stats_files = 0
    invalid_stats_files = 0

    for stats_file in stats_files:
        try:
            response = _read_json_robust(stats_file)
        except (json.JSONDecodeError, UnicodeDecodeError):
            invalid_stats_files += 1
            continue

        if not response or len(response) < 2:
            invalid_stats_files += 1
            continue

        team_stats = []
        for team_entry in response[:2]:
            stat_dict = {
                stat.get("type"): stat.get("value")
                for stat in team_entry.get("statistics", [])
                if stat.get("type") is not None
            }
            team_stats.append(stat_dict)

        valid_stats_files += 1
        all_types = set().union(*(stats.keys() for stats in team_stats))

        for stat_type in all_types:
            values = [stats.get(stat_type) for stats in team_stats]
            if all(value is not None for value in values):
                type_full_coverage[stat_type] += 1

    sorted_types = sorted(type_full_coverage.items(), key=lambda item: item[1], reverse=True)

    # ==================================================
    # OUTPUT
    # ==================================================
    line = "=" * 74
    subline = "-" * 74

    print()
    print(line)
    print(" WM 2026 DATA PIPELINE \u2013 STATUSREPORT")
    print(line)

    print("\n1. FIXTURE- UND MATCH-STATS-STATUS\n")
    print(f"{'Status':<8}{'Team':<27}{'Spiele':>9}{'Stats':>9}{'Fortschritt':>12}")
    print(subline)
    for team, games, stats, progress, status in team_rows:
        print(f"{status:<8}{team:<27}{games:>9}{stats:>9}{progress:>12}")
    print(subline)

    print("\nZusammenfassung")
    print(f"  Teams mit g\u00fcltigen Fixtures : {valid_teams:>5} / {len(WM2026_TEAMS)}")
    print(f"  Spiele inkl. Duplikate      : {total_fixtures:>5}")
    print(f"  Eindeutige Fixtures         : {len(total_unique_ids):>5}")
    print(f"  Match-Stats vorhanden       : {len(available_stats_ids):>5} / {len(total_unique_ids)}")
    print(f"  Gesamtfortschritt           : {overall_progress:>8.1f}%")

    if problem_teams:
        print("\nProblematische Teams")
        for team in problem_teams:
            print(f"  - {team}")
    else:
        print("\n\u2705 Alle Fixture-Dateien sind vorhanden und lesbar.")

    print("\n")
    print(line)
    print("2. ABDECKUNG DER STATISTIKTYPEN")
    print(line)

    print("\nZusammenfassung")
    print(f"  Gefundene Stats-Dateien     : {len(stats_files):>5}")
    print(f"  G\u00fcltige Stats-Dateien       : {valid_stats_files:>5}")
    print(f"  Leer oder ung\u00fcltig          : {invalid_stats_files:>5}")

    if valid_stats_files == 0:
        print("\nKeine g\u00fcltigen Match-Statistiken zur Analyse vorhanden.")
        print(line)
        return

    print()
    print(f"{'Statistik-Typ':<34}{'Beide Teams':>16}{'Abdeckung':>16}")
    print(subline)
    for stat_type, count in sorted_types:
        coverage = 100 * count / valid_stats_files
        if coverage >= 90:
            marker = "\u2705"
        elif coverage >= 70:
            marker = "\U0001f7e1"
        else:
            marker = "\u26a0\ufe0f"
        print(f"{marker} {stat_type:<31}{count:>16}{coverage:>15.1f}%")

    print(line)
    print()


analyze_data_status()

---

### **3) Feature-Tabelle laden**
`match_features.csv` einlesen, Datentypen setzen, sortiert nach Datum.

In [ ]:
df = pd.read_csv(FEATURES_FILE)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

print(f"Shape: {df.shape}")
print(f"Zeitraum: {df['date'].min().date()} bis {df['date'].max().date()}")
df.head()

---

### **4) Feature-Sanity-Check**
Klassenverteilung, fehlende Werte und Beispielzeilen der Feature-Tabelle.

In [ ]:
def quick_check_features(dataframe: pd.DataFrame):
    n = len(dataframe)
    cols = list(dataframe.columns)

    print("=" * 60)
    print("\u00dcBERBLICK")
    print("=" * 60)
    print(f"{'Total Zeilen:':<28} {n}")
    print(f"{'Anzahl Spalten:':<28} {len(cols)}")
    print(f"{'Spaltennamen:':<28} {', '.join(cols)}")

    print("\n" + "=" * 60)
    print("VERTEILUNG ZIELVARIABLE (result)")
    print("=" * 60)
    results = Counter(dataframe["result"])
    label_map = {"A": "Sieg Team A", "Draw": "Unentschieden", "B": "Sieg Team B"}
    for k in ("A", "Draw", "B"):
        v = results.get(k, 0)
        pct = 100 * v / n
        bar = "#" * round(pct / 2)
        print(f"  {label_map[k]:<16} {v:>5}  ({pct:5.1f}%)  {bar}")

    print("\n" + "=" * 60)
    print("FEHLENDE WERTE")
    print("=" * 60)
    missing_possession = dataframe["a_possession_avg"].isna().sum() + dataframe["b_possession_avg"].isna().sum()
    missing_elo = dataframe["elo_diff"].isna().sum()
    print(f"  {'Ohne Ballbesitz-Daten (a+b, summiert):':<40} {missing_possession:>5}")
    print(f"  {'Ohne Elo-Differenz:':<40} {missing_elo:>5}  ({100 * missing_elo / n:5.1f}%)")

    print("\n" + "=" * 60)
    print("BEISPIELZEILEN (erste 3, gruppiert nach Bereich)")
    print("=" * 60)
    meta_cols = ["date", "team_a", "team_b", "a_is_home", "goals_a", "goals_b", "result"]
    a_cols = [c for c in cols if c.startswith("a_")]
    b_cols = [c for c in cols if c.startswith("b_")]
    h2h_elo_cols = [c for c in cols if c.startswith("h2h_") or c.startswith("elo_")]

    for i, (_, r) in enumerate(dataframe.head(3).iterrows()):
        print(f"\n--- Zeile {i + 1} ---")
        print("  Spiel:   ", " | ".join(f"{c}={r[c]}" for c in meta_cols))
        print("  Team A:  ", " | ".join(f"{c.replace('a_', '')}={r[c]}" for c in a_cols))
        print("  Team B:  ", " | ".join(f"{c.replace('b_', '')}={r[c]}" for c in b_cols))
        print("  H2H/Elo: ", " | ".join(f"{c}={r[c]}" for c in h2h_elo_cols))


quick_check_features(df)

---

### **5) Klassenverteilung (Zielvariable)**
Visuelle Darstellung der Verteilung Sieg A / Unentschieden / Sieg B.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
df["result"].value_counts().reindex(["A", "Draw", "B"]).plot(
    kind="bar", ax=ax, color=["#4C72B0", "#DD8452", "#55A868"]
)
ax.set_title("Verteilung Spielausgang")
ax.set_xlabel("")
ax.set_ylabel("Anzahl Spiele")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print(df["result"].value_counts(normalize=True).round(3) * 100)

---

### **6) Feature-Verteilungen nach Ergebnis**
Zeigt, ob Kern-Features (Elo-Differenz, Formkurve) tats\u00e4chlich mit dem Ausgang zusammenh\u00e4ngen.

In [ ]:
check_cols = ["elo_diff", "a_form_points", "b_form_points"]
fig, axes = plt.subplots(1, len(check_cols), figsize=(5 * len(check_cols), 4))
for ax, col in zip(axes, check_cols):
    df.boxplot(column=col, by="result", ax=ax)
    ax.set_title(col)
    ax.set_xlabel("")
plt.suptitle("")
plt.tight_layout()
plt.show()

---

### **7) Korrelationen zwischen Features**
Hilft, redundante Features zu erkennen (z.B. falls `elo_a`/`elo_b` stark mit anderen Features korrelieren).

In [ ]:
numeric_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c != "fixture_id"]
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(numeric_cols)))
ax.set_xticklabels(numeric_cols, rotation=90)
ax.set_yticks(range(len(numeric_cols)))
ax.set_yticklabels(numeric_cols)
plt.colorbar(im, label="Korrelation")
plt.tight_layout()
plt.show()

---

### **8) Fehlende Werte pro Spalte**
\u00dcbersicht, welche Spalten wie viel Prozent fehlende Werte haben (v.a. Ballbesitz/Sch\u00fcsse/Ecken).

In [ ]:
missing_pct = (df.isna().sum() / len(df) * 100).sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 0]

if len(missing_pct) == 0:
    print("Keine fehlenden Werte.")
else:
    fig, ax = plt.subplots(figsize=(8, 0.4 * len(missing_pct) + 1))
    missing_pct.plot(kind="barh", ax=ax, color="#C44E52")
    ax.set_xlabel("% fehlende Werte")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

---

### **9) Modell-Evaluation**
L\u00e4dt das trainierte Modell und wertet es auf demselben chronologischen Test-Split aus
wie `train_model.py` (damit die Zahlen konsistent bleiben).

In [ ]:
model = joblib.load(MODELS_DIR / "random_forest_model.pkl")
imputer = joblib.load(MODELS_DIR / "imputer.pkl")
feature_cols = json.loads((MODELS_DIR / "feature_columns.json").read_text(encoding="utf-8"))

split_idx = int(len(df) * 0.8)
test_df = df.iloc[split_idx:]
X_test = test_df[feature_cols]
y_test = test_df["result"]

X_test_imp = imputer.transform(X_test)
y_pred = model.predict(X_test_imp)

print(classification_report(y_test, y_pred))

In [ ]:
labels = sorted(y_test.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)

fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels)
ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
ax.set_xlabel("Vorhergesagt")
ax.set_ylabel("Tats\u00e4chlich")
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

---

### **10) Feature Importance**
Welche Features beeinflussen die Vorhersage des Random-Forest-Modells am st\u00e4rksten?

In [ ]:
importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 6))
importances.head(15).plot(kind="barh", ax=ax, color="#4C72B0")
plt.gca().invert_yaxis()
ax.set_xlabel("Feature Importance")
ax.set_title("Top 15 wichtigste Features")
plt.tight_layout()
plt.show()

---

### **11) Einzelne Vorhersage testen**
Simuliert, was das Streamlit-Dashboard sp\u00e4ter tun wird: zwei Teams ausw\u00e4hlen, aktuelle
Formkurve/Elo/H2H berechnen (als heutiges Datum, nicht historisch), Vorhersage abrufen.

In [ ]:
from build_features import (
    load_team_id_cache, load_elo_ratings, get_elo,
    build_team_match_history, compute_rolling_features, compute_h2h,
)

team_id_cache = load_team_id_cache()
id_to_fifa_name = {v: k for k, v in team_id_cache.items()}
elo_by_year = load_elo_ratings()


def predict_matchup(team_a: str, team_b: str, as_of_date: str = "2026-07-14"):
    """Berechnet Features 'as of heute' f\u00fcr zwei Teams und gibt die
    Modell-Wahrscheinlichkeiten zur\u00fcck (Vorschau auf die Dashboard-Logik)."""
    own_id_a = team_id_cache[team_a]
    own_id_b = team_id_cache[team_b]
    hist_a = build_team_match_history(team_a, own_id_a, id_to_fifa_name)
    hist_b = build_team_match_history(team_b, own_id_b, id_to_fifa_name)

    feat_a = compute_rolling_features(hist_a, as_of_date)
    feat_b = compute_rolling_features(hist_b, as_of_date)
    if feat_a is None or feat_b is None:
        print("Zu wenig Historie f\u00fcr eines der beiden Teams.")
        return None

    h2h = compute_h2h(hist_a, team_b, as_of_date)
    year = int(as_of_date[:4])
    elo_a = get_elo(elo_by_year, team_a, year)
    elo_b = get_elo(elo_by_year, team_b, year)

    row = {"a_is_home": True}
    row.update({f"a_{k}": v for k, v in feat_a.items()})
    row.update({f"b_{k}": v for k, v in feat_b.items()})
    row.update(h2h)
    row["elo_a"] = elo_a
    row["elo_b"] = elo_b
    row["elo_diff"] = (elo_a - elo_b) if (elo_a is not None and elo_b is not None) else None

    X_new = pd.DataFrame([row])[feature_cols]
    X_new_imp = imputer.transform(X_new)
    proba = model.predict_proba(X_new_imp)[0]
    return dict(zip(model.classes_, proba))


# Beispiel - hier Teamnamen anpassen und ausprobieren:
result = predict_matchup("Canada", "Mexico")
if result:
    print("Vorhersage Canada (A) vs. Mexico (B):")
    for outcome, prob in sorted(result.items(), key=lambda x: -x[1]):
        label = {"A": "Sieg Canada", "B": "Sieg Mexico", "Draw": "Unentschieden"}[outcome]
        print(f"  {label:<16} {prob*100:5.1f}%")